# NB_hospital_daily_summary

Builds `adb_caresync_health.gold.hospital_daily_summary` (one row per hospital per calendar day with tests) per the STTM Silver to Gold v1.0. Driving table: `silver.lab_results` grouped by `hospital_id`, `test_date`; LEFT JOIN to `silver.hospitals` and `silver.calendar` (on `date_key`). Day-over-day trend via LAG(). Full rebuild.

Note: exact reload duplicates of `lab_result_id` in `silver.lab_results` are removed (latest `insert_timestamp`) before aggregation.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync_health.gold;

CREATE OR REPLACE TABLE adb_caresync_health.gold.hospital_daily_summary AS
WITH lab_dedup AS (
  SELECT *
  FROM adb_caresync_health.silver.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id ORDER BY insert_timestamp DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                                 AS summary_date,
    COUNT(*)                                  AS total_tests,
    COUNT(DISTINCT patient_id)                AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                 AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count
  FROM lab_dedup
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                              AS hospital_id,
  h.hospital_name,
  w.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(w.total_tests AS INT)                                              AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                   AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                      AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                      AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                      AS critical_test_count,
  CAST(try_divide(w.abnormal_test_count, w.total_tests) * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(try_divide(w.critical_test_count, w.total_tests) * 100 AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                     AS prev_day_total_tests,
  CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
       ELSE CAST((w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100 AS DECIMAL(6,2))
  END                                                                     AS test_volume_change_pct
FROM with_lag w
LEFT JOIN adb_caresync_health.silver.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN adb_caresync_health.silver.calendar c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT);

In [0]:

updated_df = spark.table("adb_caresync_health.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))